In [20]:
import os
from io import BytesIO
from pathlib import Path

import boto3
import dotenv
import pandas as pd
import altair as alt

In [21]:
env_path = dotenv.find_dotenv(usecwd=True)
dotenv.load_dotenv(env_path)

True

In [ ]:
COUNTY_LIST: set[str] = {
    "Romania",
    "Hungary",
    "Poland"
}
VARIABLE: str = "t2m_max"

In [23]:
def get_era5_country_averages() -> pd.DataFrame:
    s3 = boto3.client("s3")
    bucket: str = os.getenv("S3_BUCKET_NAME")
    assert bucket
    response = s3.get_object(Bucket=bucket, Key="analysis/daily-country-averages/era5.parquet")
    daily = pd.read_parquet(
        BytesIO(response["Body"].read()),
        columns=["country", "date", VARIABLE],
    )
    daily["date"] = pd.to_datetime(daily["date"])
    daily["year"] = daily["date"].dt.year
    daily = daily[daily["country"].isin(COUNTY_LIST)]
    return daily

In [24]:
daily_df: pd.DataFrame = get_era5_country_averages()

In [25]:
daily_df.head()

,country,date,t2m_max,year
97,Hungary,1950-01-01,-1.280286,1950
181,Romania,1950-01-01,-4.821524,1950
97,Hungary,1950-01-02,-0.324502,1950
181,Romania,1950-01-02,-2.626997,1950
97,Hungary,1950-01-03,3.429468,1950


In [26]:
baseline_df: pd.DataFrame = daily_df.loc[
    (daily_df["year"].between(1961, 1990)) &
    (daily_df["date"].dt.month.isin([6, 7]))
].copy()

In [27]:
baseline_df.head()

,country,date,t2m_max,year
97,Hungary,1961-06-01,25.792429,1961
181,Romania,1961-06-01,23.276005,1961
97,Hungary,1961-06-02,24.044865,1961
181,Romania,1961-06-02,25.000109,1961
97,Hungary,1961-06-03,20.857716,1961


In [28]:
# Make sure there are 30 unique years
assert baseline_df["year"].nunique() == 30

In [29]:
# Full date coverage for the baseline period
assert baseline_df["date"].nunique() == 61 * 30

In [30]:
# No missing values
assert baseline_df.isna().sum().sum() == 0

In [31]:
baseline_mean: pd.DataFrame = (
    baseline_df.groupby("country")[VARIABLE]
        .mean()
        .reset_index()
)

In [32]:
baseline_mean.head()

,country,t2m_max
0,Hungary,24.976462
1,Romania,23.660751


In [33]:
analysis_df: pd.DataFrame = (
    daily_df.loc[
        (daily_df["year"].between(1961, 2026)) &
        (daily_df["date"].dt.month.isin([6, 7]))
    ].copy()
     .groupby(["country", "year"])[VARIABLE]
     .mean()
     .reset_index()
     .merge(baseline_mean, on=["country"], how="inner", validate="many_to_one", suffixes=("", "_baseline"))
     .assign(anomaly=lambda x: x[VARIABLE] - x[f"{VARIABLE}_baseline"])
)

In [34]:
analysis_df.head()

,country,year,t2m_max,t2m_max_baseline,anomaly
0,Hungary,1961,26.077559,24.976462,1.101097
1,Hungary,1962,23.889714,24.976462,-1.086748
2,Hungary,1963,27.630426,24.976462,2.653964
3,Hungary,1964,27.343266,24.976462,2.366804
4,Hungary,1965,23.935210,24.976462,-1.041252


In [35]:
alt.Chart(analysis_df).mark_bar().encode(
    x=alt.X(
        'year:Q',
        title='Year',
        axis=alt.Axis(
            format="d",
            tickMinStep=5,
        ),
    ),
    y='anomaly:Q',
    facet='country:N',
    color='country:N'
 ).properties(width=300)

alt.Chart(...)

In [36]:
pivot_df: pd.DataFrame = analysis_df.pivot(
    index='year',
    columns='country',
    values='anomaly'
).round(3)

In [37]:
pivot_df.tail(10)

country,Hungary,Romania
year,,
2017,3.005,2.666
2018,1.626,1.339
2019,3.236,2.705
2020,0.861,1.587
2021,4.299,2.812
2022,4.302,3.725
2023,2.379,2.509
2024,3.828,5.428
2025,3.582,4.350


In [38]:
pivot_df.tail(10).mean()

country
Hungary    3.1038
Romania    2.9790
dtype: float64

Over the past decade, Hungary's June and July highs have exceeded the 1961-1990 average by 3.1 degrees Celsius (5.6 Fahrenheit), according to the Reuters Climate Monitor. Romania isn't far behind, with an averaged 3 degrees Celsius (5.4 Fahrenheit) above normal.

In [39]:
pivot_df.to_csv("pivot.csv", index=True)